# Load Dataset and find null percentage

In [31]:
import pandas as pd
%run load_data.ipynb

data = load_all_data()
print(data.keys())

for name,df in data.items():
    null_percent = (df.isnull().sum() / len(df))*100
    null_percent = null_percent[null_percent>0]
    if not null_percent.empty:
        print(f"/n _____{name}________")
        print(null_percent.sort_values(ascending = False))


dict_keys(['sugar', 'inflation', 'conflict', 'crude_oil', 'natural_gas', 'wheat', 'soya_oil', 'gold', 'silver'])
/n _____inflation________
1960           100.000000
2025           100.000000
Unnamed: 70    100.000000
1961            49.248120
1962            45.112782
                  ...    
2010             3.007519
2018             3.007519
2015             3.007519
2011             3.007519
2014             3.007519
Length: 67, dtype: float64
/n _____conflict________
ep_end_prec       100.000000
side_b_2nd         95.093458
gwno_b_2nd         95.093458
gwno_b             94.626168
side_a_2nd         84.540498
gwno_a_2nd         84.540498
ep_end_date        78.933022
territory_name     43.847352
dtype: float64


# Remove Null columns whose percentage above 70+

In [32]:
for name, df in data.items():

    # calculate percentage

    null_percent = (df.isnull().sum() / len(df))*100

    # filter range between 70 to 100
    drop_cols = null_percent[(null_percent > 70) &  (null_percent <= 100)].index

    # drop columns

    df.drop(columns=drop_cols,inplace= True)

    print(f"{name} -> Droped columns {list(drop_cols)}")


print("--------------------------------------------------------------------------------------------------")


print("Clean Null columns")


for name,df in data.items():
    print(f"-----{name}-----")
    print(df.columns)

       

        
    

   


sugar -> Droped columns []
inflation -> Droped columns ['1960', '2025', 'Unnamed: 70']
conflict -> Droped columns ['side_a_2nd', 'side_b_2nd', 'ep_end_date', 'ep_end_prec', 'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd']
crude_oil -> Droped columns []
natural_gas -> Droped columns []
wheat -> Droped columns []
soya_oil -> Droped columns []
gold -> Droped columns []
silver -> Droped columns []
--------------------------------------------------------------------------------------------------
Clean Null columns
-----sugar-----
Index(['Date', 'Sugar rate'], dtype='object')
-----inflation-----
Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969',
       '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978',
       '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987',
       '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996',
       '1997', '1998', 

*** 
Clean Conflict file

In [33]:
cols = list(data.values())[2]

print(cols.columns)

clean_conflict = cols.drop(columns=[
    
    'side_a','side_a_id',
    'side_b','side_b_id',
    'start_prec','start_prec2',
    'ep_end',
    'start_date' ,'start_date2',
    'territory_name'

    
])
clean_conflict.head()
clean_conflict.isna().sum()

Index(['conflict_id', 'location', 'side_a', 'side_a_id', 'side_b', 'side_b_id',
       'incompatibility', 'territory_name', 'year', 'intensity_level',
       'cumulative_intensity', 'type_of_conflict', 'start_date', 'start_prec',
       'start_date2', 'start_prec2', 'ep_end', 'gwno_a', 'gwno_loc', 'region',
       'version'],
      dtype='object')


conflict_id             0
location                0
incompatibility         0
year                    0
intensity_level         0
cumulative_intensity    0
type_of_conflict        0
gwno_a                  0
gwno_loc                0
region                  0
version                 0
dtype: int64

***
Clean Inflation File

In [34]:
df_inflation = data['inflation'].copy()
id_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
year_cols = [col for col in df_inflation.columns if col not in id_cols]
df_inflation = pd.melt(
    df_inflation,
    id_vars= id_cols,
    value_vars=year_cols,
    var_name= 'year',
    value_name= 'inflation rate'

)
df_inflation['year'] = df_inflation['year'].astype(int)

df_inflation = df_inflation.dropna(subset= ["inflation rate"])
df_inflation = df_inflation[['Country Name','Country Code', 'year','inflation rate']]

print(df_inflation.shape)
print(df_inflation.head())

(14038, 4)
                   Country Name Country Code  year  inflation rate
1   Africa Eastern and Southern          AFE  1961        1.933947
3    Africa Western and Central          AFW  1961        3.278446
9                     Argentina          ARG  1961       20.310698
13                    Australia          AUS  1961        3.220836
14                      Austria          AUT  1961        5.087154


### Clean Commodity Files

In [35]:


def commodity_file(df,price_col_name):
    df = df.copy()

    df['year'] = pd.to_datetime(df.iloc[:,0],format='mixed',dayfirst=True).dt.year

    df = df.rename(columns={df.columns[1]: price_col_name})

    df = df.groupby('year')[price_col_name].mean().reset_index()

    df = df.dropna()

    return df

data['sugar']       = commodity_file(data['sugar'],       'sugar_price')
data['crude_oil']   = commodity_file(data['crude_oil'],   'crude_oil_price')
data['natural_gas'] = commodity_file(data['natural_gas'], 'natural_gas_price')
data['wheat']       = commodity_file(data['wheat'],       'wheat_price')
data['soya_oil']    = commodity_file(data['soya_oil'],    'soya_oil_price')
data['gold']        = commodity_file(data['gold'],        'gold_price')
data['silver']      = commodity_file(data['silver'],      'silver_price')


for name in ['sugar', 'crude_oil', 'natural_gas', 'wheat', 'soya_oil', 'gold', 'silver']:
    print(f'{name}-> {data[name].shape}')
    print(data[name].head(2))
    print()

sugar-> (65, 2)
   year  sugar_price
0  1962     0.038825
1  1963     0.075182

crude_oil-> (81, 2)
   year  crude_oil_price
0  1946         1.357500
1  1947         1.840833

natural_gas-> (30, 2)
   year  natural_gas_price
0  1997           2.401667
1  1998           2.049167

wheat-> (68, 2)
   year  wheat_price
0  1959     1.981194
1  1960     1.955358

soya_oil-> (63, 2)
   year  soya_oil_price
0  1960        0.090727
1  1961        0.110169

gold-> (112, 2)
   year  gold_price
0  1915       19.25
1  1916       19.47

silver-> (112, 2)
   year  silver_price
0  1915          0.51
1  1916          0.67



In [36]:
df_merged = clean_conflict.copy()


df_merged = df_merged.merge(data['sugar'],       on='year', how='left')
df_merged = df_merged.merge(data['crude_oil'],   on='year', how='left')
df_merged = df_merged.merge(data['natural_gas'], on='year', how='left')
df_merged = df_merged.merge(data['wheat'],       on='year', how='left')
df_merged = df_merged.merge(data['soya_oil'],    on='year', how='left')
df_merged = df_merged.merge(data['gold'],        on='year', how='left')
df_merged = df_merged.merge(data['silver'],      on='year', how='left')


null_percent = (df_merged.isna().sum()/ len(df_merged))*100
print(null_percent[null_percent>0].sort_values(ascending = False))

natural_gas_price    59.696262
sugar_price          10.319315
soya_oil_price        8.917445
wheat_price           8.294393
dtype: float64


In [37]:
df_merged =  df_merged.drop(columns=['natural_gas_price'])

fill_col = ['sugar_price','soya_oil_price','wheat_price']
for col in fill_col:
    median_fill = df_merged[col].median()
    df_merged[col] = df_merged[col].fillna(median_fill) 


print(df_merged.isna().sum())
print(df_merged.shape)

conflict_id             0
location                0
incompatibility         0
year                    0
intensity_level         0
cumulative_intensity    0
type_of_conflict        0
gwno_a                  0
gwno_loc                0
region                  0
version                 0
sugar_price             0
crude_oil_price         0
wheat_price             0
soya_oil_price          0
gold_price              0
silver_price            0
dtype: int64
(2568, 17)


In [38]:
df_merged.head()

,conflict_id,location,incompatibility,year,intensity_level,cumulative_intensity,type_of_conflict,gwno_a,gwno_loc,region,version,sugar_price,crude_oil_price,wheat_price,soya_oil_price,gold_price,silver_price
0,11342,India,1,2012,1,0,3,750,750,3,22.1,0.215474,94.029167,7.504588,0.510109,1679.833333,31.626667
1,11342,India,1,2014,1,0,3,750,750,3,22.1,0.163502,91.560000,5.776297,0.368285,1256.250000,18.759167
2,11343,"Egypt, Israel",1,1967,2,1,2,651,"651, 666",2,22.1,0.021774,3.026667,1.599782,0.094935,35.310000,1.658250
3,11343,"Egypt, Israel",1,1969,1,1,2,651,"651, 666",2,22.1,0.034709,3.295000,1.328638,0.089652,40.839167,1.798333
4,11343,"Egypt, Israel",1,1970,1,1,2,651,"651, 666",2,22.1,0.037824,3.350833,1.534094,0.119389,35.985833,1.751667


In [39]:
print(df_inflation.head())

                   Country Name Country Code  year  inflation rate
1   Africa Eastern and Southern          AFE  1961        1.933947
3    Africa Western and Central          AFW  1961        3.278446
9                     Argentina          ARG  1961       20.310698
13                    Australia          AUS  1961        3.220836
14                      Austria          AUT  1961        5.087154


In [40]:
print(df_merged['location'].value_counts().head())

location
Myanmar (Burma)    286
India              181
Ethiopia           126
Philippines        115
Israel              77
Name: count, dtype: int64


In [41]:
single = df_merged[~df_merged['location'].str.contains(',')].shape[0]
multi  = df_merged[df_merged['location'].str.contains(',')].shape[0]

print(f"Single country: {single}")
print(f"Multi country:  {multi}")


Single country: 2426
Multi country:  142


In [42]:
country_mapping = {
    'Myanmar (Burma)'         : 'Myanmar',
    'Zimbabwe (Rhodesia)'     : 'Zimbabwe',
    'DR Congo (Zaire)'        : 'Congo, Dem. Rep.',
    'Russia (Soviet Union)'   : 'Russia',
    'Serbia (Yugoslavia)'     : 'Serbia',
    'Vietnam (North Vietnam)' : 'Vietnam',
    'Madagascar (Malagasy)'   : 'Madagascar',
    'Cambodia (Kampuchea)'    : 'Cambodia',
    'Yemen (North Yemen)'     : 'Yemen, Rep.',
    'South Yemen'             : 'Yemen, Rep.',
    'Ivory Coast'             : "Cote d'Ivoire",
    'Bosnia-Herzegovina'      : 'Bosnia and Herzegovina',
    'United States of America': 'United States',
    'Turkey'                  : 'Turkiye',
    'Iran'                    : 'Iran, Islamic Rep.',
    'Syria'                   : 'Syrian Arab Republic',
    'Egypt'                   : 'Egypt, Arab Rep.',
    'Venezuela'               : 'Venezuela, RB',
    'Congo'                   : 'Congo, Rep.',
}

df_merged = df_merged.copy()
df_merged['country'] = df_merged['location'].str.split(',').str[0].str.strip()
df_merged['country'] = df_merged['country'].replace(country_mapping)

print(df_merged[['location', 'country']].head())

        location           country
0          India             India
1          India             India
2  Egypt, Israel  Egypt, Arab Rep.
3  Egypt, Israel  Egypt, Arab Rep.
4  Egypt, Israel  Egypt, Arab Rep.


In [43]:
# Dekho kaun se countries match nahi hui
conflict_countries = set(df_merged['country'].dropna().unique())
inflation_countries = set(df_inflation['Country Name'].dropna().unique())
not_matched = conflict_countries - inflation_countries

print(f"Match nahi hui countries: {len(not_matched)}")
print(sorted(not_matched))

Match nahi hui countries: 10
['Brunei', 'Gambia', 'Hyderabad', 'Kyrgyzstan', 'Laos', 'North Korea', 'Russia', 'Somalia', 'South Vietnam', 'Vietnam']


In [44]:
df_inflation = df_inflation.copy()
df_final = df_merged.merge(
    df_inflation[['Country Name', 'year', 'inflation rate']],
    left_on=['country', 'year'],
    right_on=['Country Name', 'year'],
    how='left'
)

df_final = df_final.drop(columns=['Country Name', 'version'])

total = len(df_final)
nan_count = df_final['inflation rate'].isnull().sum()
percentage = (nan_count / total) * 100

print(df_final.shape)
print(f"NaN count: {nan_count}")
print(f"NaN %: {percentage:.2f}%")

(2568, 18)
NaN count: 492
NaN %: 19.16%


In [45]:
median_inflation = df_final['inflation rate'].median()
df_final['inflation rate'] = df_final['inflation rate'].fillna(median_inflation)

print(f"NaN after imputation: {df_final['inflation rate'].isnull().sum()}")
print(f"Final shape: {df_final.shape}")
print(df_final.head(3))

NaN after imputation: 0
Final shape: (2568, 18)
   conflict_id       location  incompatibility  year  intensity_level  \
0        11342          India                1  2012                1   
1        11342          India                1  2014                1   
2        11343  Egypt, Israel                1  1967                2   

   cumulative_intensity  type_of_conflict gwno_a  gwno_loc region  \
0                     0                 3    750       750      3   
1                     0                 3    750       750      3   
2                     1                 2    651  651, 666      2   

   sugar_price  crude_oil_price  wheat_price  soya_oil_price   gold_price  \
0     0.215474        94.029167     7.504588        0.510109  1679.833333   
1     0.163502        91.560000     5.776297        0.368285  1256.250000   
2     0.021774         3.026667     1.599782        0.094935    35.310000   

   silver_price           country  inflation rate  
0     31.626667      

In [46]:
df_final.to_csv("../Data/Processed_data/war_impact_clean.csv", index=False)
print(f"Saved! Shape: {df_final.shape}")
print(df_final.columns.tolist())

Saved! Shape: (2568, 18)
['conflict_id', 'location', 'incompatibility', 'year', 'intensity_level', 'cumulative_intensity', 'type_of_conflict', 'gwno_a', 'gwno_loc', 'region', 'sugar_price', 'crude_oil_price', 'wheat_price', 'soya_oil_price', 'gold_price', 'silver_price', 'country', 'inflation rate']
